# Práctica 3 — Modelo final con scikit-learn

## Objetivo y regla anti-fuga

Construir y ajustar de forma explícita el modelo elegido en P2. Primero se reserva un test reproducible; después, **todo transformador que aprende de los datos** se ajusta únicamente dentro de un `Pipeline` sobre entrenamiento. La validación cruzada y el ajuste de hiperparámetros no consultan el test, que se evalúa una sola vez al final.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


## 1. Carga y definición del problema

No hace falta ajustar una ruta: el notebook localiza la raíz del repositorio desde su directorio actual. Ajusta, si procede, la variable objetivo. `rating_high` es el ejemplo de clasificación usado en P2. Excluye identificadores, variables que no existirían en el momento de predecir y cualquier columna que revele el objetivo.


In [ ]:
def find_project_root() -> Path:
    """Locate the repository from the current Jupyter working directory."""
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pixi.toml").is_file() and (candidate / "03-machine-learning").is_dir():
            return candidate
    raise FileNotFoundError(
        "No se ha podido localizar la raíz del repositorio. Abre el notebook dentro del repositorio PIA."
    )


PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / "03-machine-learning/03-practicas/midterm-capstone/Peliculas/Practica_Peliculas_OK/data/movies.csv"
if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"No se encuentra {DATA_PATH}. Solicita el movies.csv validado y guárdalo en esa ruta."
    )

df = pd.read_csv(DATA_PATH)
TARGET = "rating_high"

# Defensa adicional: el contrato no las emite. Si aparecieran, son IDs,
# métricas posteriores o la fuente directa de la etiqueta y no pueden predecir.
FORBIDDEN_PREDICTORS = [
    "id", "tmdb_id", "imdb_id", "title", "imdb_rating", "vote_average", "vote_count",
]
DROP_COLUMNS = [column for column in FORBIDDEN_PREDICTORS if column in df.columns]

X = df.drop(columns=[TARGET, *DROP_COLUMNS])
y = df[TARGET]
X.shape, y.value_counts(dropna=False)

## 2. Reserva de test antes de aprender transformaciones

Esta separación se hace **antes** de imputar, codificar o escalar. `stratify=y` conserva aproximadamente la proporción de clases y `random_state=42` permite reproducir el experimento.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

X_train.shape, X_test.shape


## 3. Preprocesado dentro del pipeline

Las columnas se identifican a partir de **`X_train`**, no de todos los datos. Los valores faltantes, la codificación y el escalado se aprenden en cada partición de entrenamiento de la validación cruzada. Así no se filtra información del test ni de la parte de validación de cada fold.


In [ ]:
numeric_features = X_train.select_dtypes(include="number").columns.tolist()
categorical_features = X_train.select_dtypes(exclude="number").columns.tolist()

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)
categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features),
    ],
    remainder="drop",
)

pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(random_state=42, class_weight="balanced")),
    ]
)
pipeline


## 4. Ajuste con validación cruzada: solo entrenamiento

Elige y justifica la métrica. Aquí se usa F1 por coherencia con P2. `GridSearchCV` recibe únicamente `X_train` e `y_train`; cada fold ajusta una copia completa del pipeline.


In [ ]:
param_grid = {
    "model__n_estimators": [200, 400],
    "model__max_depth": [None, 10, 20],
    "model__min_samples_leaf": [1, 2, 5],
}

search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1,
    refit=True,
)
search.fit(X_train, y_train)

cv_results = pd.DataFrame(search.cv_results_).sort_values("rank_test_score")
cv_results[["rank_test_score", "mean_test_score", "std_test_score", "params"]].head()


## 5. Decisión antes de consultar el test

Documenta el mejor conjunto de hiperparámetros y el resultado medio de CV. Si decides cambiar variables, métrica o espacio de búsqueda, hazlo aquí y vuelve a validar solo con entrenamiento. **No ejecutes aún ninguna predicción sobre `X_test`.**


In [ ]:
best_pipeline = search.best_estimator_
print("Mejor F1 media de CV:", search.best_score_)
print("Hiperparámetros:", search.best_params_)


## 6. Evaluación final única sobre test reservado

Cuando hayas cerrado la decisión, ejecuta esta sección una sola vez. Las métricas y el análisis de errores describen el comportamiento sobre datos no usados ni para preparar ni para ajustar el modelo. No reutilices este test para decidir una nueva iteración.


In [ ]:
y_pred = best_pipeline.predict(X_test)

final_metrics = pd.Series(
    {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
    }
)
final_metrics


In [ ]:
print(classification_report(y_test, y_pred, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
plt.show()

error_analysis = X_test.copy()
error_analysis["actual"] = y_test
error_analysis["predicted"] = y_pred
errors = error_analysis.loc[error_analysis["actual"] != error_analysis["predicted"]]
errors.head(10)


## 7. Conclusiones y trazabilidad

Incluye: variables excluidas y motivo; columnas numéricas/categóricas; métrica y por qué; configuración de CV; mejores hiperparámetros; métricas finales; dos o tres patrones observados en los errores; limitaciones del dataset y una mejora futura.

Si tras ver el test decides modificar el modelo, declara este test agotado y reserva otro conjunto para evaluar la nueva versión.


## Anexo GPU (opcional)

Una alternativa GPU puede acelerar el entrenamiento, pero no cambia la regla: la partición, los transformadores y la validación se diseñan con entrenamiento; el test se consulta una vez al final.


In [ ]:
# Ejemplo a investigar y adaptar al entorno disponible:
# from cuml.ensemble import RandomForestClassifier as RF_GPU
